# `dsfb-robotics` — reproducible Colab notebook

**Reads the residuals existing robotics observers already produce, structures them into a typed grammar (Admissible / Boundary / Violation), and emits a deterministic, reviewable timeline alongside — never replacing — the incumbent pipeline's own output.**

## What this notebook does

From a fresh `git clone`, this notebook:

1. Installs Rust 1.85.1 (Colab) or skips if already present.
2. Builds the `paper-lock` release binary with the canonical `lto=true, codegen-units=1` profile.
3. Verifies bit-exact determinism (three back-to-back runs, SHA-256 identical).
4. Runs `paper-lock` on every one of the **20 real-world processed-slice datasets** committed at `data/processed/<slug>.csv`. **No `--fixture` flag, no synthetic, no simulation data anywhere in this notebook.**
5. Renders the full structural-figure set the paper ships — every `\includegraphics` referenced in `paper/dsfb_robotics.tex` — inline.
6. Computes Cohen's $d$ on the slate-wide bimodality (zero-Violation cluster vs non-zero-Violation cluster).
7. Demonstrates the operator-facing `--explain` and `--emit-review-csv` flags.
8. Bundles every artefact (JSON outputs, figures, effect-size CSV, review CSV, the notebook itself) into a single timestamped zip and (on Colab) opens a download dialog.

## Run-All time budget

| Phase | Free-tier Colab (2 vCPU) | Local laptop (cold) | Local (cached `target/`) |
|---|---|---|---|
| Cargo build (`--release`, LTO) | ~12 min | ~6 min | ~5 s |
| `paper-lock` × 20 datasets | ~2 min | ~30 s | ~30 s |
| Figure rendering (24 PDFs+PNGs) | ~10 min | ~3 min | ~3 min |
| Audits (effect size, explain, review CSV) | ~30 s | ~5 s | ~5 s |
| Artefact bundle (zip + sha256) | ~30 s | ~10 s | ~10 s |
| **Total** | **~30 min** | **~10 min** | **~4 min** |

Well under the free-tier 12-hour session limit.

## Hard non-claim

Existing methods continue to outperform DSFB at their own tasks. DSFB does not detect faults earlier, classify them, or replace any incumbent observer. It recovers structure from residuals those methods discard, as a strictly read-only side-channel. See `docs/DEPLOYMENT_ANTIPATTERNS.md` and the paper §10.21 for the full discipline.


## §2 · Mathematical primer

DSFB is the composition of three small ideas. Read this section once; everything below operationalises it.

### 2.1 The residual sign tuple

Every production robotics observer (Kalman / Luenberger / inverse-dynamics / whole-body MPC / centroidal-momentum / vibration RMS) computes a residual stream $r(k) \in \mathbb{R}^d$ as a by-product. Incumbent practice treats this residual as a scalar discrepancy to minimise. DSFB lifts it into a three-dimensional **sign tuple**:

$$\sigma(k) = \bigl(\,\|r(k)\|,\ \dot{r}(k),\ \ddot{r}(k)\,\bigr)$$

- $\|r(k)\|$ — instantaneous magnitude.
- $\dot{r}(k)$ — drift rate, the rolling first difference.
- $\ddot{r}(k)$ — slew, the rolling second difference.

The sign tuple is the semiotic-manifold coordinate. DSFB's grammar is a partition of this 3-D space into typed states.


### 2.2 Stage III calibration: ρ = μ + 3σ

From the first 20 % of finite samples (the commissioning window), DSFB computes the empirical mean and standard deviation of $\|r(k)\|$ and freezes one scalar:

$$\rho = \mu_{[0,N/5]} + 3\,\sigma_{[0,N/5]}.$$

This is the **admissibility envelope** — the residual magnitude DSFB considers compatible with healthy operation. It is computed once per dataset, never re-tuned during the run. The 20 % / $3\sigma$ choice is the canonical Stage III posture and matches the protocol pre-registered at git tag `paper-lock-protocol-frozen-v1`.

Anti-pattern: tuning $\rho$ at deployment time to hit a target review surface destroys cross-deployment comparability. See `docs/DEPLOYMENT_ANTIPATTERNS.md` anti-pattern 3.


### 2.3 Grammar FSM (Admissible / Boundary / Violation)

Given $\sigma(k)$ and $\rho$, the FSM commits to one of three grammar states, with a 2-confirmation hysteresis (so single-sample noise never flips the state):

| State | Magnitude band | Structural meaning |
|---|---|---|
| **Admissible** | $\|r\| \le \beta\rho$ | Residual sits cleanly inside the envelope. |
| **Boundary** | $\beta\rho < \|r\| \le \rho$ with one of the reason-codes below | Operator review surface. |
| **Violation** | $\|r\| > \rho$ | Envelope exit; structurally consequential. |

**Boundary reason codes** (`src/grammar.rs`): `sustained-outward-drift` (drift positive across a window of $W=8$ samples), `abrupt-slew` ($|\ddot r| > \delta_s$), `recurrent-grazing` ($\|r\| > \beta\rho$ on $K=4$ of the last $W$ samples).

**Canonical parameters** (frozen at git tag `paper-lock-protocol-frozen-v1`):

$$W = 8,\quad K = 4,\quad \beta = 0.5,\quad \delta_s = 0.05.$$

The augmentation thesis: same residual, same incumbent conclusion, an additional human-readable structural layer. DSFB does not write back to the controller, the observer, or the safety chain. It is a strictly read-only side-channel. See the paper §1.4 hero figure for the single-glance visual.


## §3 · Environment setup

Detects Colab vs local. On Colab: clones the repo, installs Rust 1.85.1 via rustup. On local: skips both if already present. The pinned toolchain version comes from `rust-toolchain.toml` — no version drift possible.


In [ ]:
import os
import pathlib
import shutil
import subprocess
import sys

ON_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/infinityabundance/dsfb.git"

# Locate or clone the crate.
candidates = [
    pathlib.Path("dsfb-robotics"),
    pathlib.Path("dsfb/crates/dsfb-robotics"),
    pathlib.Path("/home/one/dsfb/crates/dsfb-robotics"),
]
CRATE_DIR = next((c for c in candidates if c.is_dir()), None)

if CRATE_DIR is None:
    if ON_COLAB:
        subprocess.run(["git", "clone", "--depth=1", REPO_URL, "dsfb"], check=True)
        CRATE_DIR = pathlib.Path("dsfb/crates/dsfb-robotics").resolve()
    else:
        raise SystemExit("crate not found; clone github.com/infinityabundance/dsfb first")
else:
    CRATE_DIR = CRATE_DIR.resolve()

print(f"crate root: {CRATE_DIR}")
assert (CRATE_DIR / "Cargo.toml").is_file(), "missing Cargo.toml — wrong directory"
assert (CRATE_DIR / "data" / "processed").is_dir(), "missing data/processed/ — wrong directory"

# Install Rust if missing (rust-toolchain.toml pins 1.85.1; rustup honours it).
if shutil.which("cargo") is None:
    print("installing Rust toolchain via rustup ...")
    subprocess.run(
        ["sh", "-c", "curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y --default-toolchain none"],
        check=True,
    )
    os.environ["PATH"] = f"{os.path.expanduser('~')}/.cargo/bin:{os.environ['PATH']}"

subprocess.run(["rustc", "--version"], check=True, cwd=CRATE_DIR)


## §4 · Build the `paper-lock` release binary

`cargo build --release --bin paper-lock --features std,paper_lock` builds the binary under the canonical release profile from `Cargo.toml`:

```toml
[profile.release]
lto = true              # whole-program optimisation
codegen-units = 1       # single unit → deterministic codegen
panic = "abort"         # no unwinding, smaller code
```

LTO + single codegen-unit is **slow** (~12 min on free-tier Colab, ~6 min on a developer laptop) but produces a binary that emits **bit-identical output across architectures**. That cross-architecture byte-equality is the determinism falsifier (paper §12.5): any divergence falsifies the determinism claim. See `audit/cross_arch/` for evidence.

On a laptop with a cached `target/` directory the build is a no-op (~5 seconds).


In [ ]:
result = subprocess.run(
    ["cargo", "build", "--release", "--bin", "paper-lock", "--features", "std,paper_lock"],
    cwd=str(CRATE_DIR), check=False,
)
if result.returncode != 0:
    raise SystemExit(f"cargo build failed (exit {result.returncode})")

BIN = CRATE_DIR / "target" / "release" / "paper-lock"
assert BIN.is_file(), f"binary not produced at {BIN}"
print(f"binary: {BIN}")


## §5 · Bit-exact determinism gate

Runs `paper-lock cwru` three times against the literal real-world processed CSV at `data/processed/cwru.csv`, SHA-256s each stdout JSON, and asserts all three are bit-identical. Then queries `paper-lock --list` to confirm the canonical 20-slug roster.

Falsifier (paper §12.5 *Falsifiability*): any byte-divergent re-run on bit-exact input falsifies the determinism claim of the engine. This cell is the determinism-falsifier check, executed in your runtime.


In [ ]:
import hashlib

def run_paperlock(*args: str) -> str:
    """Run paper-lock from the crate root and return stdout (real data; no --fixture)."""
    r = subprocess.run([str(BIN), *args], cwd=str(CRATE_DIR),
                       check=True, capture_output=True, text=True)
    return r.stdout

h = [hashlib.sha256(run_paperlock("cwru").encode()).hexdigest() for _ in range(3)]
print("sha256 across 3 runs:")
for i, sh in enumerate(h, 1):
    print(f"  run {i}: {sh}")
assert h[0] == h[1] == h[2], "determinism gate FAILED — output drifted across runs"
print("  ✓ bit-identical across 3 invocations")

slugs = run_paperlock("--list").strip().split()
print(f"\n--list returned {len(slugs)} slugs:")
print("  " + ", ".join(slugs))
assert len(slugs) == 20, f"expected 20 datasets, got {len(slugs)}"


## §6 · Run `paper-lock` on every real-world dataset (no `--fixture`)

Drives the loop from `paper-lock --list` (matches the CI workflow at `.github/workflows/reproduce.yml`). For each slug, invokes the production binary with `--emit-episodes` so the per-sample episode trace is included in the JSON. **The notebook never passes `--fixture`** — every result below comes from the literal real-world residual stream at `data/processed/<slug>.csv`. The four kinematic-arm rows additionally consume `*_published.csv` (the literal published-θ residual computed by the upstream identification model).

Output: a tab-aligned per-dataset census table (slug, N, A, B, V, compression, V/N).


In [ ]:
import json

# JSON outputs land at audit/json_outputs/<slug>.json — the canonical
# location consumed by scripts/effect_size.py and audit/checksums.txt.
JSON_OUT = CRATE_DIR / "audit" / "json_outputs"
JSON_OUT.mkdir(parents=True, exist_ok=True)

reports: dict[str, dict] = {}
for slug in slugs:
    out = run_paperlock(slug, "--emit-episodes")
    (JSON_OUT / f"{slug}.json").write_text(out)
    reports[slug] = json.loads(out)

print(f"{'slug':<28s}{'N':>8s}{'A':>8s}{'B':>8s}{'V':>8s}{'comp':>8s}{'V/N':>8s}")
print("-" * 76)
for slug in sorted(reports):
    agg = reports[slug]["aggregate"]
    n   = agg["total_samples"]
    a   = agg["admissible"]
    b   = agg["boundary"]
    v   = agg["violation"]
    cr  = agg["compression_ratio"]
    vr  = v / n if n else 0.0
    print(f"{slug:<28s}{n:>8d}{a:>8d}{b:>8d}{v:>8d}{cr:>8.3f}{vr:>8.3f}")

n_zero_v   = sum(1 for r in reports.values() if r['aggregate']['violation'] == 0)
n_total    = len(reports)
print(f"\n{n_zero_v} of {n_total} datasets are zero-Violation (silent-augment cluster — correct posture).")
print(f"  JSON outputs: {JSON_OUT}")


## §7 · Render the full per-dataset figure set

The crate's figure pipeline at `scripts/figures_real.py` ships seven rendering functions that together produce **every figure cited by the paper** (24 unique PDFs):

| Function | Output filename | Scope |
|---|---|---|
| `render_grammar_timeline` | `grammar_timeline.{pdf,png}` | every slug |
| `render_envelope` | `residual_on_envelope.{pdf,png}` | every slug |
| `render_semiotic_manifold` | `semiotic_manifold_3d.{pdf,png}` | every slug (3-D scatter of $\sigma(k)$) |
| `render_per_joint_3d` | `per_joint_3d.{pdf,png}` | arm rows: `kuka_lwr`, `panda_gaz`, `dlr_justin`, `ur10_kufieta` |
| `render_comparison` | `comparison.{pdf,png}` | exemplars: `cwru`, `kuka_lwr`, `icub_pushrecovery` |
| `render_hero_augmentation` | `hero_augmentation.{pdf,png}` | `panda_gaz` only — the §1.4 hero side-by-side panel on the literal Gaz-2019 published-θ residual |
| `render_compression_histogram` | `_all_compression_histogram.{pdf,png}` | one cross-slate summary, family-coloured |

We invoke `figures_real.py` as a script (no `--no-3d` flag, so the 3-D semiotic manifolds and per-joint scatters render). It loads the real CSVs at `data/processed/`, runs the canonical FSM offline (matching `paper-lock`), and writes every figure to `paper/figures/<slug>/`.

**The notebook does not modify the figure pipeline.** It re-uses the production renderers exactly as they ship.


In [ ]:
FIG_OUT = CRATE_DIR / "paper" / "figures"
FIG_OUT.mkdir(parents=True, exist_ok=True)

result = subprocess.run(
    [sys.executable, str(CRATE_DIR / "scripts" / "figures_real.py")],
    cwd=str(CRATE_DIR), capture_output=True, text=True,
)
if result.returncode != 0:
    print("stderr (last 40 lines):")
    print("\n".join(result.stderr.splitlines()[-40:]))
    raise SystemExit(f"figures_real.py failed (exit {result.returncode})")

# Print the per-dataset summary lines from stderr (matches local invocation).
for line in result.stderr.splitlines():
    if line.startswith("[figures_real]"):
        print(line)
print(f"\nfigure tree: {FIG_OUT}")


### §7a · PHM family (bearing / run-to-failure)

Three datasets where the residual is a vibration health-index trajectory: `cwru` (CWRU Bearing — seeded inner-race fault), `ims` (NASA IMS Run-to-Failure), `femto_st` (FEMTO-ST PHM-2012 challenge run-to-failure). The expected motif is **BpfiGrowth**: ball-pass-frequency-inner amplitude trending upward as the bearing degrades. See `src/heuristics.rs::RoboticsMotif::BpfiGrowth`.


In [ ]:
from IPython.display import Image, Markdown, display

def show(slug: str, name: str, caption: str = "") -> None:
    p = FIG_OUT / slug / f"{name}.png"
    if not p.is_file():
        print(f"  (missing: {p.relative_to(CRATE_DIR)})")
        return
    if caption:
        display(Markdown(f"**{slug}** — {caption}"))
    display(Image(filename=str(p)))

for slug in ["cwru", "ims", "femto_st"]:
    show(slug, "grammar_timeline", "grammar-state timeline")
    show(slug, "residual_on_envelope", "residual norm vs the Stage-III envelope ρ")
    show(slug, "semiotic_manifold_3d", "σ(k) = (‖r‖, ṙ, r̈) sign-tuple cloud, coloured by grammar state")
show("cwru", "comparison", "incumbent 3σ threshold (top) vs DSFB grammar (bottom) on the same residual")


### §7b · Kinematic arm rows (literal published-θ residuals)

Four manipulator datasets where the residual is the inverse-dynamics identification residual computed against the published model:

- `panda_gaz` — Franka Panda, Gaz 2019 cpp dynamic model (IEEE RA-L 4(4): 4147–4154, DOI 10.1109/LRA.2019.2931248).
- `dlr_justin` — DLR-class Panda link-side-torque, Giacomuzzo 2024 (Zenodo 12516500).
- `ur10_kufieta` — UR10, Pinocchio RNEA on URSim URDF (Kufieta 2014, Polydoros 2015).
- `kuka_lwr` — KUKA LWR-IV+ Simionato 7R kinematic-domain residual.

Expected motifs: **StribeckGap** (low-velocity friction non-linearity producing a torque-residual plateau-then-drop), **BacklashRing** (oscillatory residual at velocity reversals from harmonic-drive backlash). See `src/heuristics.rs`.


In [ ]:
for slug in ["panda_gaz", "dlr_justin", "ur10_kufieta", "kuka_lwr"]:
    show(slug, "grammar_timeline", "grammar timeline on the literal published-model residual")
    show(slug, "residual_on_envelope")
    show(slug, "per_joint_3d", "per-joint 3-D residual scatter — reveals which joint(s) carry the structure")
    show(slug, "semiotic_manifold_3d")
show("kuka_lwr", "comparison", "incumbent threshold vs DSFB grammar on the same residual")


### §7c · Manipulation and teleop kinematics

Eight teleoperation / manipulation rows with kinematic-domain residuals (DROID, Open X-Embodiment, ALOHA family, Mobile ALOHA, SO-ARM100). Many of these are silent-augment by design: the residual stream is dominated by the Admissible regime and DSFB correctly contributes minimal structural surface — that is the framework's correct behaviour on clean teleop data, not a failure.


In [ ]:
manipulation = [
    "droid", "openx", "so100",
    "aloha_static", "aloha_static_tape", "aloha_static_screw_driver",
    "aloha_static_pingpong_test", "mobile_aloha",
]
for slug in manipulation:
    show(slug, "grammar_timeline")
    show(slug, "semiotic_manifold_3d")


### §7d · Balancing rows (legged + humanoid)

Five whole-body-control / balancing datasets:

- `cheetah3` — MIT Cheetah 3 / Mini-Cheetah open logs (Katz 2019).
- `icub_pushrecovery` — ergoCub Romualdi 2024 ZMP-tracking residual (literal published controller; IEEE Humanoids 2024).
- `icub3_sorrentino` — ergoCub Sorrentino 2025 deliberately-perturbed balancing trial (IEEE RAL 2025).
- `anymal_parkour` — ANYmal-C GrandTour outdoor locomotion (ETH Zürich Legged Robotics 2024).
- `unitree_g1` — Unitree G1 humanoid teleoperation.

Expected motifs: **GrfDesync** (commanded vs measured ground-reaction-force divergence), **MpcStanceLag** (whole-body MPC contact force arriving late vs measured foot contact), **CoMDrift** (centroidal-momentum-observer estimate drifting vs the model — the load-bearing balancing motif).


In [ ]:
for slug in ["cheetah3", "icub_pushrecovery", "icub3_sorrentino", "anymal_parkour", "unitree_g1"]:
    show(slug, "grammar_timeline")
    show(slug, "residual_on_envelope")
    show(slug, "semiotic_manifold_3d")
show("icub_pushrecovery", "comparison", "ZMP-tracking residual: incumbent threshold (top) vs DSFB grammar (bottom)")


### §7e · The §1.4 hero augmentation panel (panda_gaz)

The single-glance figure for the augmentation thesis. Top panel: the Gaz-2019 pipeline collapses the entire residual stream to one trajectory-aggregate $\sigma_{\text{noise}}$ scalar (drawn as a horizontal blue band) over the faintly-overlaid residual trace. Bottom panel: DSFB structures the same per-timestep residual into a typed grammar timeline.

Same residual. Same incumbent conclusion. An additional human-readable layer.


In [ ]:
show("panda_gaz", "hero_augmentation",
     "top: one σ_noise scalar (Gaz incumbent) over the residual; bottom: DSFB grammar on the same trace")


## §8 · Cross-slate compression histogram

Per-dataset review-surface compression ratio $(B + V) / N$, coloured by family (PHM grey / Kinematics blue / Balancing purple). The slate is genuinely heterogeneous: range $[0.10, 0.94]$, coefficient of variation $\approx 0.54$. Compression is a *structural fingerprint*, not a quality metric — see `docs/DEPLOYMENT_ANTIPATTERNS.md` anti-pattern 2.


In [ ]:
show_global = lambda name: display(Image(filename=str(FIG_OUT / f"{name}.png")))
show_global("_all_compression_histogram")


## §9 · Effect size — bimodality of the slate

Cohen's $d$ on the per-dataset Violation rate $V/N$, partitioned by $V > 0$ vs $V = 0$. The zero-Violation cluster is the silent-augment posture — DSFB correctly contributes minimal structural surface on the cleanly-operating rows.

Conventional Cohen's $d$ thresholds: small $d{=}0.2$, medium $d{=}0.5$, large $d{=}0.8$.


In [ ]:
result = subprocess.run(
    [sys.executable, str(CRATE_DIR / "scripts" / "effect_size.py")],
    cwd=str(CRATE_DIR), capture_output=True, text=True, check=True,
)
print(result.stdout)


## §10 · Operator-facing flags

Two flags that turn `paper-lock` from a paper-reproduction tool into an operator-facing triage instrument.

### 10.1 `--explain` — post-commit narrative

Adds an `explain[]` array to the JSON output. Each entry is a one-line narrative for one Boundary or Violation episode: which structural condition fired the FSM transition, the sample index, the residual band. **Read this as a triage prompt, not a fault explanation.** The narrative says *why the FSM committed*, not *why the robot is broken*; physical interpretation is the operator's job. See `docs/DEPLOYMENT_ANTIPATTERNS.md` anti-pattern 4.


In [ ]:
doc = json.loads(run_paperlock("panda_gaz", "--explain"))
narratives = doc.get("explain", [])
print(f"explain[] entries: {len(narratives)}")
for entry in narratives[:5]:
    print(json.dumps(entry, indent=2))
if len(narratives) > 5:
    print(f"  ... ({len(narratives) - 5} more entries; full array in the JSON output)")


### 10.2 `--emit-review-csv` — operator handoff

Writes one row per Boundary / Violation episode in a spreadsheet-friendly schema: `start, end, length, label, reason_code, peak_norm`. Suitable for piping straight into a triage queue or dashboard. The CSV is independent of (and narrower than) the JSON output; the JSON remains the canonical schema-validated artefact.


In [ ]:
csv_path = pathlib.Path("/tmp/panda_gaz_review.csv")
subprocess.run(
    [str(BIN), "panda_gaz", "--emit-review-csv",
     "--review-csv-path", str(csv_path)],
    cwd=str(CRATE_DIR), check=True, capture_output=True,
)
print(f"wrote {csv_path}  ({csv_path.stat().st_size:,} bytes)")
with csv_path.open() as f:
    for i, line in enumerate(f):
        if i >= 11: break
        print(line.rstrip())


## §11 · Bundle every artefact and download

Bundles everything this notebook produced into a single zip — `paper-lock` JSON outputs, every per-dataset figure (PDF + PNG), the cross-slate compression histogram, the effect-size cluster CSV, the operator-facing review CSV, and the executed notebook itself. Prints the SHA-256 of the zip so you can compare across runs and architectures (any divergence is a determinism falsifier; see paper §12.5).

On Colab the bundle automatically opens a download dialog. On a local Jupyter you'll see the zip path printed; copy it out manually.


In [ ]:
import hashlib as _hashlib
import zipfile
from datetime import datetime, timezone

stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
ZIP_PATH = pathlib.Path(f"/tmp/dsfb_robotics_artifacts_{stamp}.zip")

def add_tree(zf: zipfile.ZipFile, root: pathlib.Path, arc_root: str) -> int:
    """Add every file under root into the zip under arc_root/. Returns count."""
    n = 0
    if not root.is_dir():
        return 0
    for p in sorted(root.rglob("*")):
        if p.is_file():
            zf.write(p, arcname=f"{arc_root}/{p.relative_to(root)}")
            n += 1
    return n

with zipfile.ZipFile(ZIP_PATH, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    n_json = add_tree(zf, JSON_OUT, "audit/json_outputs")
    n_fig  = add_tree(zf, FIG_OUT, "paper/figures")
    n_eff  = add_tree(zf, CRATE_DIR / "audit" / "effect_size", "audit/effect_size")
    review_csv = pathlib.Path("/tmp/panda_gaz_review.csv")
    if review_csv.is_file():
        zf.write(review_csv, arcname="operator/panda_gaz_review.csv")
    # Include the executed notebook itself for full provenance.
    nb_path = CRATE_DIR / "colab" / "dsfb_robotics_reproduce.ipynb"
    if nb_path.is_file():
        zf.write(nb_path, arcname="colab/dsfb_robotics_reproduce.ipynb")
    # README inside the zip describing what's in it.
    manifest = (
        f"dsfb-robotics — Colab reproduction artefact bundle\n"
        f"timestamp (UTC):  {stamp}\n"
        f"crate root:       {CRATE_DIR}\n"
        f"paper-lock JSONs: {n_json}\n"
        f"figure files:     {n_fig}\n"
        f"effect-size CSVs: {n_eff}\n"
        f"\nThis bundle is produced by colab/dsfb_robotics_reproduce.ipynb.\n"
        f"It does NOT include raw data (only processed-residual outputs).\n"
        f"Every figure here was rendered from the real-world processed-slice CSVs.\n"
    )
    zf.writestr("MANIFEST.txt", manifest)

size_mb = ZIP_PATH.stat().st_size / (1024 * 1024)
sha = _hashlib.sha256(ZIP_PATH.read_bytes()).hexdigest()
print(f"bundle:  {ZIP_PATH}")
print(f"size:    {size_mb:.2f} MiB")
print(f"sha256:  {sha}")
print(f"entries: {n_json + n_fig + n_eff} files (+ MANIFEST.txt + notebook)" )

if ON_COLAB:
    try:
        from google.colab import files as colab_files
        colab_files.download(str(ZIP_PATH))
        print("  → Colab download dialog opened.")
    except Exception as e:
        print(f"  (Colab download skipped: {e}; copy the path above manually)")
else:
    print("  (local Jupyter: copy the zip path above)")


## §12 · Honesty, non-claims, and next steps

### Four disclosures (re-stated for the reviewer who jumps straight to this cell)

1. **No outperforms claim.** DSFB does not detect faults earlier than threshold alarms / CUSUM / EWMA / RMS, does not classify bearing faults, does not predict RUL, does not replace any incumbent observer. Existing methods continue to outperform DSFB at their own tasks.
2. **Silent-augment is correct behaviour.** Seven of twenty datasets land in the zero-Violation cluster. On clean teleoperation, healthy bearings, well-controlled balancing, the residual is dominated by the Admissible regime — DSFB correctly contributes minimal structural surface. That is a feature, not a failure.
3. **`--explain` is a triage prompt, not a fault diagnosis.** The narrative names the structural condition that fired the FSM transition. Physical root-cause attribution is the operator's job, using domain expertise the framework cannot provide.
4. **Real-world data only.** All 20 datasets are physical-hardware recordings under permissive licences. Zero synthetic, zero simulation, zero MuJoCo / Isaac / Gazebo / RaiSim / Drake / Webots / PyBullet. C-MAPSS (aerospace simulation) and Franka Kitchen (MuJoCo) were removed during review.

### Where to go from here

- **Local one-command reproduction:** `bash scripts/reproduce.sh` — fetches data (when possible), preprocesses, runs `paper-lock` on every dataset, regenerates audit checksums, compiles the paper.
- **30-minute deep dive:** [`docs/REVIEWER_GUIDE.md`](../docs/REVIEWER_GUIDE.md) — minute-by-minute walkthrough.
- **Audit stack:** Miri (3 alias models), Kani (6 harnesses), cargo-fuzz (1 M iter × 2 targets), DSFB-gray (96.2 % strong assurance), bootstrap CIs, sensitivity / ablation grids — all under `audit/`.
- **The paper:** [`paper/dsfb_robotics.tex`](../paper/dsfb_robotics.tex) — 69 pages, every empirical number reproducible from this notebook.
- **Citation:** [`CITATION.cff`](../CITATION.cff).

### Determinism / cross-architecture replay

If you run this notebook on a different architecture and your Section §5 SHA-256 hashes differ from a published reference, that is a falsifier of the determinism claim in paper §12.5. File an issue with `audit/checksums.fresh.txt` and your platform tuple.
